# Iterative insight analysis

This notebook analyzes canonical Skill Drilla artifacts from the validation bundle and surfaces recurring insight candidates in stages.

Guardrails:
- consume canonical artifacts under `artifacts/chat-analysis/validation/full-smoke`
- preserve canonical `evidence_id`, `session_id`, `project_id`, and recurrence semantics
- treat any notebook-derived thresholds, grouping, and synthesis as exploratory notebook analysis
- treat semantic clustering only as an optional **non-canonical** overlay


In [1]:
from collections import Counter, defaultdict
from pathlib import Path

from skill-drilla.notebooks import (
    load_corpus_view,
    load_detector_run,
    load_normalization_diagnostics,
    load_parse_diagnostics,
    load_report_metadata,
    load_semantic_run,
    load_validation_summary,
    read_json,
)

REPO_ROOT = Path('.')
ARTIFACT_ROOT = REPO_ROOT / 'artifacts' / 'chat-analysis'
VALIDATION_ROOT = ARTIFACT_ROOT / 'validation' / 'full-smoke'
SEMANTIC_RUN_PATH = ARTIFACT_ROOT / 'semantic' / 'clustering-smoke' / 'semantic_run.json'

TOP_N_FINDINGS = 12
TOP_N_EVIDENCE_PER_FINDING = 5
MIN_DISTINCT_SESSIONS = 20
MIN_DISTINCT_PROJECTS = 3
COMPARE_SUBAGENT_VIEW = True
ENABLE_SEMANTIC_OVERLAY = True


In [ ]:
validation_summary = load_validation_summary(VALIDATION_ROOT / 'validation_summary.json')
discovery_summary = read_json(VALIDATION_ROOT / 'discovery' / 'inventory_summary.json')
parse_diagnostics = load_parse_diagnostics(VALIDATION_ROOT / 'parse' / 'parse_diagnostics.json')
normalization_diagnostics = load_normalization_diagnostics(VALIDATION_ROOT / 'normalize' / 'normalization_diagnostics.json')
report_metadata = load_report_metadata(VALIDATION_ROOT / 'reports' / 'report_metadata.json')
primary_view = load_corpus_view(VALIDATION_ROOT / 'views' / 'user_nl_root_only')
comparison_view = load_corpus_view(VALIDATION_ROOT / 'views' / 'combined_nl_root_plus_subagent')
detector_runs = {
    detector_name: load_detector_run(Path(metadata['detector_run']))
    for detector_name, metadata in validation_summary['detectors'].items()
}

{
    'validation_root': str(VALIDATION_ROOT),
    'primary_view': primary_view['view_name'],
    'primary_included_rows': primary_view['summary']['included_record_count'],
    'comparison_view': comparison_view['view_name'],
    'comparison_included_rows': comparison_view['summary']['included_record_count'],
    'detector_count': len(detector_runs),
    'report_section_count': report_metadata['report_scope']['finding_count'],
    'scoped_sessions': discovery_summary['scoped_sessions'],
    'unknown_record_shapes': parse_diagnostics['aggregate']['unknown_record_shapes'],
    'ambiguous_items': len(normalization_diagnostics['ambiguous_items']),
}


In [ ]:
{
    'primary_description': primary_view['summary']['description'],
    'primary_counts': primary_view['summary']['counts'],
    'primary_subagent_policy': primary_view['summary']['subagent_policy'],
    'comparison_description': comparison_view['summary']['description'],
    'comparison_counts': comparison_view['summary']['counts'],
    'comparison_subagent_policy': comparison_view['summary']['subagent_policy'],
    'row_expansion_factor': round(
        comparison_view['summary']['included_record_count'] / primary_view['summary']['included_record_count'],
        2,
    ),
    'recurrence_basis': primary_view['summary']['recurrence_basis'],
}


In [ ]:
def flatten_sections(report_metadata):
    return sorted(report_metadata['sections'], key=lambda section: section['rank'])


def evidence_index_from_sections(sections):
    indexed = defaultdict(list)
    for section in sections:
        for evidence_id in section['evidence_ids']:
            indexed[evidence_id].append(section)
    return indexed


def detector_summary_rows(sections):
    grouped = defaultdict(list)
    for section in sections:
        grouped[(section['detector'], section['category'])].append(section)

    rows = []
    for (detector, category), items in sorted(grouped.items()):
        rows.append({
            'detector': detector,
            'category': category,
            'section_count': len(items),
            'total_score': round(sum(item['score'] for item in items), 2),
            'avg_distinct_sessions': round(sum(item['recurrence']['distinct_sessions'] for item in items) / len(items), 2),
            'avg_distinct_projects': round(sum(item['recurrence']['distinct_projects'] for item in items) / len(items), 2),
            'direct_user_supported_sections': sum(1 for item in items if item['direct_user_evidence_ids']),
        })
    return sorted(rows, key=lambda row: (-row['total_score'], -row['avg_distinct_sessions'], row['detector']))


def shortlist_sections(sections, *, min_distinct_sessions, min_distinct_projects, limit):
    shortlisted = [
        section for section in sections
        if section['recurrence']['distinct_sessions'] >= min_distinct_sessions
        and section['recurrence']['distinct_projects'] >= min_distinct_projects
    ]
    return shortlisted[:limit]


sections = flatten_sections(report_metadata)
evidence_to_sections = evidence_index_from_sections(sections)
section_detector_summary = detector_summary_rows(sections)

len(sections), len(evidence_to_sections), section_detector_summary[:3]


In [ ]:
top_sections = sections[:TOP_N_FINDINGS]
[
    {
        'rank': section['rank'],
        'heading': section['heading'],
        'detector': section['detector'],
        'category': section['category'],
        'score': section['score'],
        'distinct_sessions': section['recurrence']['distinct_sessions'],
        'raw_occurrences': section['recurrence']['raw_occurrences'],
        'distinct_projects': section['recurrence']['distinct_projects'],
        'sample_sources': [reference['source_anchor'] for reference in section['source_references'][:3]],
        'notebook_interpretation': (
            'broadly recurring'
            if section['recurrence']['distinct_sessions'] >= 100
            else 'more concentrated but still notable'
        ),
    }
    for section in top_sections
]


In [ ]:
section_detector_summary[:TOP_N_FINDINGS]


In [ ]:
reused_evidence = sorted(
    (
        {
            'evidence_id': evidence_id,
            'referencing_sections': len(linked_sections),
            'headings': [section['heading'] for section in linked_sections[:3]],
            'detectors': sorted({section['detector'] for section in linked_sections}),
        }
        for evidence_id, linked_sections in evidence_to_sections.items()
        if len(linked_sections) > 1
    ),
    key=lambda row: (-row['referencing_sections'], row['evidence_id']),
)

reused_evidence[:TOP_N_FINDINGS]


In [ ]:
{
    'comparison_enabled': COMPARE_SUBAGENT_VIEW,
    'primary_view_name': primary_view['view_name'],
    'comparison_view_name': comparison_view['view_name'],
    'primary_counts': primary_view['summary']['counts'],
    'comparison_counts': comparison_view['summary']['counts'],
    'raw_occurrence_delta': comparison_view['summary']['counts']['raw_occurrences'] - primary_view['summary']['counts']['raw_occurrences'],
    'distinct_evidence_delta': comparison_view['summary']['counts']['distinct_evidence'] - primary_view['summary']['counts']['distinct_evidence'],
    'scope_note': 'Comparison view expands to assistant natural language and subagent sessions while preserving canonical recurrence definitions.',
}


In [ ]:
{
    'scoped_sessions': discovery_summary['scoped_sessions'],
    'excluded_sessions': discovery_summary['excluded_sessions'],
    'lineage_coverage': discovery_summary['lineage_coverage'],
    'parse_failures': parse_diagnostics['aggregate'],
    'top_inclusion_status_counts': dict(list(normalization_diagnostics['inclusion_status_counts'].items())[:4]),
    'top_semantic_class_counts': dict(list(normalization_diagnostics['semantic_class_counts'].items())[:6]),
    'ambiguous_item_count': len(normalization_diagnostics['ambiguous_items']),
    'guardrail_note': 'Interpret findings in the context of excluded subagent sessions, filtered defaults, and unknown record shapes.',
}


In [10]:
semantic_overlay = None
if ENABLE_SEMANTIC_OVERLAY and SEMANTIC_RUN_PATH.exists():
    semantic_overlay = load_semantic_run(SEMANTIC_RUN_PATH)

if semantic_overlay is None:
    {'semantic_overlay': 'disabled or unavailable'}
else:
    cluster_lookup = {}
    for cluster in semantic_overlay['derived_output'].get('clusters', []):
        for evidence_id in cluster['evidence_ids']:
            cluster_lookup[evidence_id] = cluster['label']

    overlay_rows = []
    for section in top_sections:
        labels = [cluster_lookup[evidence_id] for evidence_id in section['evidence_ids'] if evidence_id in cluster_lookup]
        overlay_rows.append({
            'heading': section['heading'],
            'non_canonical': semantic_overlay['non_canonical'],
            'cluster_count': len(set(labels)),
            'sample_clusters': sorted(set(labels))[:5],
        })

    overlay_rows


In [ ]:
shortlisted = shortlist_sections(
    sections,
    min_distinct_sessions=MIN_DISTINCT_SESSIONS,
    min_distinct_projects=MIN_DISTINCT_PROJECTS,
    limit=TOP_N_FINDINGS,
)

[
    {
        'heading': section['heading'],
        'detector': section['detector'],
        'score': section['score'],
        'distinct_sessions': section['recurrence']['distinct_sessions'],
        'distinct_projects': section['recurrence']['distinct_projects'],
        'direct_user_evidence_count': len(section['direct_user_evidence_ids']),
        'sample_source_anchors': [reference['source_anchor'] for reference in section['source_references'][:TOP_N_EVIDENCE_PER_FINDING]],
        'caveats': section['caveats'],
    }
    for section in shortlisted
]
